In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

RUN_LABEL = "fresh_rerun"
REPO_ROOT = ""
GIT_CLONE_URL = "https://github.com/dttdrv/gerhard.git"
GIT_CLONE_BRANCH = "main"
AUTO_MOUNT_DRIVE = True
AUTO_DOWNLOAD_BUNDLE = True
SEED = 42
DISTILL_STEPS_OVERRIDE = ""
BATCH_SIZE_OVERRIDE = ""
ACCUMULATION_STEPS_OVERRIDE = ""
EVAL_INTERVAL_OVERRIDE = ""


def _run(cmd, cwd=None, env=None, check=True, capture_output=False):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def _ensure_jupyter():
    try:
        _run([sys.executable, "-m", "jupyter", "--version"])
    except Exception:
        _run([sys.executable, "-m", "pip", "install", "-q", "jupyter", "nbconvert", "ipykernel"])


def _mount_drive_if_needed():
    if "google.colab" in sys.modules and AUTO_MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)


def _find_repo_root() -> Path:
    candidates = []
    if REPO_ROOT:
        candidates.append(Path(REPO_ROOT).expanduser())
    candidates.extend([
        Path("/content/gerhard"),
        Path("/content/drive/MyDrive/gerhard"),
        Path.cwd(),
        *Path.cwd().parents,
    ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (
            (candidate / "notebooks" / "asnn_goose_colab_v15.ipynb").exists()
            and (candidate / "scripts" / "register_dossier_run.py").exists()
        ):
            return candidate
    target = Path("/content/gerhard")
    if target.exists():
        shutil.rmtree(target)
    _run(["git", "clone", "--branch", GIT_CLONE_BRANCH, GIT_CLONE_URL, str(target)])
    return target.resolve()


def _gpu_name() -> str:
    try:
        result = _run([
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader",
        ], capture_output=True)
        return result.stdout.strip().splitlines()[0].strip()
    except Exception:
        return "unknown"


def _default_training_overrides(gpu_name: str):
    upper = gpu_name.upper()
    if "T4" in upper:
        return {"batch_size": 4, "accumulation_steps": 4, "eval_interval": 300}
    if "L4" in upper:
        return {"batch_size": 8, "accumulation_steps": 2, "eval_interval": 300}
    return {"batch_size": 4, "accumulation_steps": 2, "eval_interval": 300}


def _git_commit(repo_root: Path) -> str:
    try:
        result = _run(["git", "rev-parse", "HEAD"], cwd=str(repo_root), capture_output=True)
        return result.stdout.strip()
    except Exception:
        return "unknown"


_mount_drive_if_needed()
_ensure_jupyter()
repo_root = _find_repo_root()
training_notebook = repo_root / "notebooks" / "asnn_goose_colab_v15.ipynb"
if not training_notebook.exists():
    raise FileNotFoundError(f"Missing training notebook: {training_notebook}")

gpu_name = _gpu_name()
training_defaults = _default_training_overrides(gpu_name)
batch_size = int(BATCH_SIZE_OVERRIDE) if str(BATCH_SIZE_OVERRIDE).strip() else training_defaults["batch_size"]
accumulation_steps = int(ACCUMULATION_STEPS_OVERRIDE) if str(ACCUMULATION_STEPS_OVERRIDE).strip() else training_defaults["accumulation_steps"]
eval_interval = int(EVAL_INTERVAL_OVERRIDE) if str(EVAL_INTERVAL_OVERRIDE).strip() else training_defaults["eval_interval"]

run_id = f"v15_{RUN_LABEL}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
operator_dir = repo_root / "outputs" / "operator_runs" / run_id
operator_dir.mkdir(parents=True, exist_ok=True)
executed_notebook_name = f"{run_id}_training_executed.ipynb"

git_commit = _git_commit(repo_root)
print(f"GPU: {gpu_name}")
print(f"Repo root: {repo_root}")
print(f"Training notebook: {training_notebook}")
print(f"Run ID: {run_id}")
print(f"Batch size: {batch_size}")
print(f"Accumulation steps: {accumulation_steps}")
print(f"Eval interval: {eval_interval}")
print(f"GIT_COMMIT: {git_commit}")

exec_env = os.environ.copy()
exec_env.update(
    {
        "GERHARD_RUN_ID": run_id,
        "GERHARD_SEED": str(SEED),
        "GERHARD_BATCH_SIZE": str(batch_size),
        "GERHARD_ACCUMULATION_STEPS": str(accumulation_steps),
        "GERHARD_EVAL_INTERVAL": str(eval_interval),
        "GERHARD_ENABLE_REGISTER_RUN": "0",
        "GERHARD_ENABLE_AUTODOWNLOAD_DOSSIER": "0",
        "GERHARD_NOTEBOOK_PATH": str(training_notebook),
        "GIT_COMMIT": git_commit,
    }
)
if str(DISTILL_STEPS_OVERRIDE).strip():
    exec_env["GERHARD_DISTILL_STEPS"] = str(DISTILL_STEPS_OVERRIDE).strip()

(operator_dir / "operator_env.json").write_text(
    json.dumps({k: exec_env[k] for k in sorted(exec_env) if k.startswith("GERHARD_") or k == "GIT_COMMIT"}, indent=2),
    encoding="utf-8",
)

cmd = [
    sys.executable,
    "-m",
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    str(training_notebook),
    "--ExecutePreprocessor.timeout=-1",
    "--output",
    executed_notebook_name,
    "--output-dir",
    str(operator_dir),
]

result = _run(cmd, cwd=str(repo_root), env=exec_env, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Fresh rerun notebook execution failed with exit code {result.returncode}")

run_dir = repo_root / "outputs" / run_id
expected = [
    "eval_suite.json",
    "metrics.json",
    "config.yaml",
    "seed.txt",
    "v15_spikingbrain.json",
    f"run_dossier_{run_id}.html",
]
optional = [
    "v15_best.pt",
    "results.json",
]
missing = [name for name in expected if not (run_dir / name).exists()]

print(f"
Artifact dir: {run_dir}")
print(f"Executed notebook: {operator_dir / executed_notebook_name}")
print("
Artifacts:")
for name in expected + optional:
    path = run_dir / name
    print(f"- {name}: {'present' if path.exists() else 'missing'} -> {path}")

if missing:
    print("
Structural stop: missing expected artifacts.")
    print("Do not register this run until you inspect the partial bundle.")
else:
    dossier_path = run_dir / f"run_dossier_{run_id}.html"
    archive_base = operator_dir / f"{run_id}_bundle"
    bundle_zip = shutil.make_archive(str(archive_base), "zip", run_dir)
    print("
Bundle ready.")
    print(f"ZIP: {bundle_zip}")
    print("Laptop-side registration command:")
    print(f'python scripts/register_dossier_run.py --dossier "{dossier_path}" --phase B')
    if AUTO_DOWNLOAD_BUNDLE and "google.colab" in sys.modules:
        from google.colab import files
        files.download(bundle_zip)
